# Structured Workflows as Alternatives to Full Autonomy

Not every problem needs a full autonomous agent loop. The loop from NB03 is maximally flexible — the model decides which tools to call, in what order, for how many turns — but that flexibility comes at a cost: unpredictable latency, hard-to-audit behavior, and a failure mode where the model spins in circles. For many real tasks, a simpler, more structured workflow is more reliable, cheaper, and far easier to debug.

Anthopic's ["Building Effective Agents"](https://www.anthropic.com/research/building-effective-agents) identifies five workflow patterns of increasing complexity — from simple prompt chaining to orchestrator-worker. This notebook implements four of them as standalone Python functions: **reflection**, **routing**, **parallelization**, and **orchestrator-worker**. We skip prompt chaining since it reduces to ordinary sequential function calls and requires no coordination logic.

These patterns are building blocks, not competing frameworks. Each solves a concrete class of problems. They also compose: an orchestrator-worker pipeline might use reflection inside each worker, or route inputs before dispatching them to parallel workers. Understanding them separately makes it much easier to reason about the combinations.

## The Pattern Spectrum

We can characterize the four patterns along four axes — number of LLM calls, latency profile, relative cost, and the class of problem each handles best:

| Pattern | LLM calls | Latency | Cost | Best For |
|---------|-----------|---------|------|---------|
| Routing | 1 + 1 handler | Low | Low | Mixed input types, specialization |
| Parallelization | $N$ concurrent | Low | $N\times$ | Decomposable, independent tasks |
| Orchestrator-worker | 1 + $N$ workers | Medium | High | Complex tasks, unknown structure |
| Reflection | $N$ iterative | High | High | Quality-critical output |

:::{.callout-important}
We skip **prompt chaining** — it's just sequential function calls, no special coordination needed. The patterns below all require real orchestration: a router making decisions, parallel coroutines, a planner writing subtasks, or an evaluator driving revision cycles.

:::

## Setup

**Setup.** Imports and client initialization against OpenRouter:

In [ ]:
import os
import json
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()

client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
)
MODEL = "anthropic/claude-sonnet-4"

All four patterns share a single non-streaming helper. We define it once here and reuse it throughout:

In [ ]:
async def chat(system: str, user: str, temperature: float = 0.7) -> str:
    """Non-streaming single-turn chat completion."""
    response = await client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

## Pattern 1 — Reflection

Reflection — also called the **evaluator-optimizer** pattern — improves output quality through iterative self-critique. A generator LLM produces an initial draft; an evaluator LLM critiques it; the generator revises; this cycle repeats until the evaluator approves or a maximum iteration count is reached. The key insight is that evaluation is often much easier than generation: it is simpler to check whether a function handles edge cases than to write a function that handles them in the first place.

**Setup.** We define separate system prompts for the generator and evaluator. The evaluator uses a special stop word `APPROVED` to signal that no further revision is needed:

In [ ]:
GENERATOR_SYSTEM = """You are a Python developer. Write clean, correct Python functions.
Include a docstring and handle edge cases. Return only the code, no surrounding text or markdown."""

EVALUATOR_SYSTEM = """You are a senior Python code reviewer. Evaluate the provided Python function.
If the code is correct, handles edge cases well, and is production quality, respond with exactly: APPROVED
Otherwise, provide specific, actionable feedback on what needs to change. Focus on correctness, not style."""

**Generator.** The reflection loop runs generate → evaluate → revise until the evaluator approves or `max_iterations` is exhausted:

In [ ]:
async def reflection_loop(
    task: str,
    max_iterations: int = 4,
) -> tuple[str, int]:
    """Improve an output through generate-evaluate cycles.

    Returns (final_output, iterations_used).
    """
    output = await chat(GENERATOR_SYSTEM, task, temperature=0.7)  # <1>

    for i in range(max_iterations):
        feedback = await chat(
            EVALUATOR_SYSTEM,
            f"Task: {task}\n\nCode:\n{output}",
            temperature=0.0,  # <2>
        )

        if "APPROVED" in feedback:  # <3>
            return output, i + 1

        revision_prompt = (
            f"Task: {task}\n\nYour previous attempt:\n{output}\n\n"
            f"Reviewer feedback:\n{feedback}\n\n"
            f"Rewrite the function addressing all the feedback."
        )
        output = await chat(GENERATOR_SYSTEM, revision_prompt, temperature=0.5)

    return output, max_iterations

1. Use non-zero temperature for generation — variety is good here, and the evaluator will filter poor outputs.
2. Zero temperature for the evaluator — quality judgments should be deterministic.
3. Check for `APPROVED` anywhere in the response, in case the evaluator adds brief commentary alongside the stop word.

Running the reflection loop on a non-trivial task:

In [ ]:
TASK = (
    "Write a function `is_palindrome(s: str) -> bool` that checks if a string is a palindrome. "
    "Handle empty strings, spaces, and mixed case."
)

print(f"Task: {TASK}\n")
result, iters = await reflection_loop(TASK)
print(f"Completed in {iters} iteration(s)\n")
print("Final result:")
print(result)

:::{.callout-note}
Reflection is powerful but expensive — each iteration costs two LLM calls. Use it when: (a) quality matters more than latency, (b) evaluation is easier than generation (code is a good fit — running tests is simpler than writing them), and (c) you have a clear quality criterion the evaluator can check.

:::

## Pattern 2 — Routing

Routing classifies an incoming request and dispatches it to a specialized handler. This improves quality — a prompt tuned for math problems consistently outperforms a general-purpose prompt on arithmetic — and can reduce cost if the router sends simple queries to cheaper prompts or smaller models. The router itself is just one deterministic LLM call.

**Setup.** We define a router prompt that classifies inputs into one of three categories, and a handler map that pairs each category with a specialized system prompt:

In [ ]:
ROUTER_SYSTEM = """Classify the user's request into exactly one category.

Categories:
- "math" — arithmetic, algebra, or quantitative calculations
- "code" — writing, debugging, or explaining code
- "general" — everything else (facts, opinions, conversation)

Return only the category name: math, code, or general. No other text."""

HANDLERS = {
    "math":    "You are a precise mathematics assistant. Always show your work step by step.",
    "code":    "You are a senior software engineer. Write correct, idiomatic code with explanations.",
    "general": "You are a helpful assistant. Be concise and accurate.",
}

**Router.** We classify the message, look up the appropriate handler, and dispatch:

In [ ]:
async def route_and_handle(message: str) -> tuple[str, str]:
    """Route a message to the appropriate handler.

    Returns (category, response).
    """
    category = await chat(ROUTER_SYSTEM, message, temperature=0.0)  # <1>
    category = category.strip().lower()
    if category not in HANDLERS:
        category = "general"  # <2>
    system = HANDLERS[category]
    response = await chat(system, message)
    return category, response

1. Temperature `0.0` for the router — classification should be deterministic.
2. Graceful fallback: if the model returns an unexpected category, treat it as general.

Testing the router on four diverse inputs:

In [ ]:
test_messages = [
    "What is 17 × 23?",
    "Write a function to reverse a linked list.",
    "What is the capital of France?",
    "Is Python faster than Rust for web servers?",
]

for msg in test_messages:
    category, response = await route_and_handle(msg)
    print(f"Query:    {msg!r}")
    print(f"Category: {category}")
    print(f"Response: {response[:200]}")
    print()

:::{.callout-note}
Routing is the cheapest pattern — one extra LLM call to classify, then standard handling. Use it when: (a) inputs vary widely in type or complexity, (b) specialized prompts outperform a single general prompt for your use case, or (c) you want to route simple queries to a cheaper model and complex ones to a more capable one.

:::

## Pattern 3 — Parallelization

Parallelization runs multiple LLM calls concurrently using `asyncio.gather`. Two sub-patterns cover most use cases: (1) **sectioning**, where a large task is split into independent parts that are processed in parallel and then combined; and (2) **voting**, where the same question is posed multiple times at non-zero temperature and the answers are aggregated to improve reliability.

### Sectioning

**Parallel summarizer.** We build a function that summarizes a list of texts concurrently, collecting coroutines first and launching them all at once:

In [ ]:
async def parallel_summarize(texts: list[str]) -> list[str]:
    """Summarize each text concurrently."""
    SUMMARIZER = "Summarize the following text in 2-3 sentences. Be concise."
    tasks = [chat(SUMMARIZER, text) for text in texts]  # <1>
    return await asyncio.gather(*tasks)                  # <2>

1. Build a list of coroutines — they have not started yet at this point.
2. `asyncio.gather` starts all coroutines concurrently and waits for all to finish, returning results in the same order as the inputs.

**Benchmark.** We compare sequential vs. parallel wall-clock time on five short texts:

In [ ]:
import time

TEXTS = [
    "Python is a high-level programming language emphasizing readability. Guido van Rossum created it in 1991. It supports multiple programming paradigms.",
    "Machine learning is a subset of artificial intelligence. It enables systems to learn from data. Common algorithms include neural networks and decision trees.",
    "Git is a distributed version control system. Linus Torvalds created it in 2005. It tracks changes to files and enables collaboration.",
    "Docker is a containerization platform. It packages applications with their dependencies. Containers are lightweight and portable.",
    "Kubernetes orchestrates containerized applications. Google open-sourced it in 2014. It automates deployment, scaling, and management.",
]

# Sequential
t0 = time.perf_counter()
sequential_results = []
for text in TEXTS:
    summary = await chat("Summarize the following text in 2-3 sentences.", text)
    sequential_results.append(summary)
sequential_time = time.perf_counter() - t0

# Parallel
t0 = time.perf_counter()
parallel_results = await parallel_summarize(TEXTS)
parallel_time = time.perf_counter() - t0

print(f"Sequential: {sequential_time:.2f}s")
print(f"Parallel:   {parallel_time:.2f}s")
print(f"Speedup:    {sequential_time / parallel_time:.1f}×")

### Voting

**Parallel vote.** We pose the same question $n$ times concurrently at non-zero temperature, then use a secondary LLM call to synthesize the answers into a single aggregated response:

In [ ]:
async def parallel_vote(question: str, n: int = 3) -> str:
    """Get N independent answers and synthesize into a majority response."""
    answers = await asyncio.gather(*[  # <1>
        chat("Answer the following question concisely.", question, temperature=0.7)
        for _ in range(n)
    ])

    aggregator = await chat(  # <2>
        "You are given multiple answers to the same question. Synthesize them into a single best answer.",
        f"Question: {question}\n\nAnswers:\n" + "\n".join(f"{i+1}. {a}" for i, a in enumerate(answers)),
        temperature=0.0,
    )
    return aggregator

1. Run $n$ independent calls concurrently — each sees the same question but, at temperature `0.7`, may generate a different answer.
2. A final deterministic call synthesizes the $n$ responses into one coherent answer.

Testing the voting approach:

In [ ]:
question = "What are the top 3 benefits of using type hints in Python?"
aggregated = await parallel_vote(question, n=3)
print(f"Question: {question}\n")
print(f"Aggregated answer:\n{aggregated}")

:::{.callout-note}
Voting implements self-consistency (NB02) with concurrency. Sequential self-consistency takes $N \times$ latency; parallel voting takes approximately $1 \times$ latency — just the slowest individual call. Use it when reliability matters and you have spare API budget.

:::

## Pattern 4 — Orchestrator-Worker

The orchestrator-worker pattern handles complex tasks whose subtasks cannot be enumerated in advance. An orchestrator LLM receives the high-level task and dynamically decomposes it into subtasks; workers execute those subtasks in parallel; a synthesizer LLM combines the results into a coherent output. Unlike prompt chaining — which uses a static sequence of steps — the orchestrator's decomposition is determined at runtime.

**Setup.** We define three system prompts: one for the orchestrator that decomposes the task into a JSON array, one template for workers that focuses each worker on its specific angle, and one for the synthesizer that writes the final report:

In [ ]:
ORCHESTRATOR_SYSTEM = """You are a task decomposition agent. Break complex research tasks into independent subtasks.

Output a JSON array of subtasks. Each subtask has:
- "id": short identifier (e.g. "python_frameworks")
- "task": the specific research question
- "focus": what angle to investigate

Return ONLY the JSON array, no other text."""

WORKER_SYSTEM_TEMPLATE = """You are a research assistant. Answer the following research question concisely and accurately.
Focus on: {focus}
Write 3-5 bullet points."""

SYNTHESIZER_SYSTEM = """You are a technical writer. You receive multiple research reports on related topics.
Synthesize them into a coherent, structured comparison report with clear headings."""

**Pipeline.** The three-step pipeline — decompose, dispatch in parallel, synthesize — is wired together in a single `async` function:

In [ ]:
async def orchestrator_pipeline(task: str) -> str:
    """Decompose a task, run workers in parallel, synthesize results."""
    # Step 1: Orchestrator decomposes
    decomposition_json = await chat(ORCHESTRATOR_SYSTEM, task, temperature=0.3)
    try:
        subtasks = json.loads(decomposition_json)
    except json.JSONDecodeError:
        import re  # <1>
        match = re.search(r'\[.*\]', decomposition_json, re.DOTALL)
        subtasks = json.loads(match.group()) if match else []

    print(f"Decomposed into {len(subtasks)} subtask(s):")
    for s in subtasks:
        print(f"  - {s['id']}: {s['task']}")

    # Step 2: Workers run in parallel
    async def run_worker(subtask: dict) -> tuple[str, str]:
        system = WORKER_SYSTEM_TEMPLATE.format(focus=subtask.get("focus", "general"))
        result = await chat(system, subtask["task"])
        return subtask["id"], result

    worker_outputs = await asyncio.gather(*[run_worker(s) for s in subtasks])  # <2>

    # Step 3: Synthesize
    reports = "\n\n".join(f"## {name}\n{report}" for name, report in worker_outputs)
    synthesis = await chat(
        SYNTHESIZER_SYSTEM,
        f"Task: {task}\n\nResearch reports:\n{reports}",
    )
    return synthesis

1. Graceful fallback: if the model wraps the JSON in surrounding text, we extract the array with a regex before parsing.
2. Workers run concurrently — the orchestrator does not wait for each worker to finish before starting the next.

Running the pipeline on a multi-faceted comparison task:

In [ ]:
research_task = (
    "Compare Python and TypeScript for building web APIs: "
    "consider developer experience, performance, ecosystem, and type safety."
)

synthesis = await orchestrator_pipeline(research_task)
print(f"\n{'=' * 60}")
print(synthesis)

:::{.callout-note}
The orchestrator-worker pattern is more flexible than a static pipeline but harder to control. The orchestrator's decomposition can be surprising — always inspect the subtask list. This pattern is the foundation for multi-agent systems (NB09), where each worker is itself a full agent with its own tool registry and memory.

:::

## Choosing the Right Pattern

Each pattern has a natural home. The table below provides a quick decision guide:

| Pattern | Use when | Avoid when |
|---------|----------|------------|
| Reflection | Quality > speed; evaluation is easy | Task is simple; cost is constrained |
| Routing | Inputs vary widely in type or complexity | All inputs are similar; no specialization needed |
| Parallelization | Task decomposes into independent parts | Steps are sequential and interdependent |
| Orchestrator-worker | Complex task; subtasks unknown in advance | Simple task; overhead not justified |

:::{.callout-note}
The autonomous agent from NB03 subsumes all of these — it is the most flexible but least predictable option. Prefer a structured pattern when: (a) you can enumerate the steps in advance, (b) you need predictable latency, or (c) you need to audit exactly what happened. Use the autonomous agent when the task is open-ended, the steps are unknowable in advance, or maximum flexibility is required.

:::

---

■